# 04 - Signal Quality Scoring

Create a 0-1 quality score from missing percentage, variance, peak consistency, noise, and amplitude stability.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

quality_schema = T.StructType([
    T.StructField('missing_score', T.DoubleType(), True),
    T.StructField('variance_score', T.DoubleType(), True),
    T.StructField('peak_consistency_score', T.DoubleType(), True),
    T.StructField('noise_score', T.DoubleType(), True),
    T.StructField('amplitude_stability_score', T.DoubleType(), True),
    T.StructField('signal_quality_score', T.DoubleType(), True),
])

def clamp(value, lower=0.0, upper=1.0):
    return float(max(lower, min(upper, value)))

@F.udf(quality_schema)
def score_signal_quality(signal, missing_fraction, sampling_rate):
    import numpy as np

    arr = np.asarray(signal or [], dtype=float)
    if arr.size == 0:
        arr = np.zeros(1, dtype=float)

    missing_component = clamp(1.0 - float(missing_fraction or 0.0))
    std_value = float(np.std(arr))
    variance_component = 0.0 if std_value < 0.02 else clamp(std_value / 0.25)

    peaks = []
    minimum_distance = max(1, int(0.30 * int(sampling_rate or 100)))
    last_peak = -minimum_distance
    threshold = float(np.mean(arr) + 0.15 * np.std(arr))
    for index in range(1, arr.size - 1):
        if arr[index] > arr[index - 1] and arr[index] > arr[index + 1] and arr[index] >= threshold:
            if index - last_peak >= minimum_distance:
                peaks.append(index)
                last_peak = index

    if len(peaks) < 2:
        peak_component = 0.35
    else:
        intervals = np.diff(np.asarray(peaks))
        interval_cv = float(np.std(intervals) / (np.mean(intervals) + 1e-8))
        peak_component = clamp(1.0 - interval_cv / 0.60)

    roughness = float(np.mean(np.abs(np.diff(arr))) / (np.std(arr) + 1e-8)) if arr.size > 2 else 0.5
    noise_component = clamp(1.0 - (roughness - 0.15) / 1.25)

    signal_range = float(np.max(arr) - np.min(arr))
    if signal_range <= 1e-8:
        amplitude_component = 0.0
    else:
        iqr = float(np.percentile(arr, 75) - np.percentile(arr, 25))
        amplitude_component = clamp(1.0 - abs((iqr / signal_range) - 0.45) / 0.45)

    total_score = (
        0.25 * missing_component
        + 0.20 * variance_component
        + 0.25 * peak_component
        + 0.15 * noise_component
        + 0.15 * amplitude_component
    )

    return (missing_component, variance_component, peak_component, noise_component, amplitude_component, clamp(total_score))

clean_df = spark.table('cardio_ppg_clean')
quality_result_df = clean_df.withColumn(
    'quality_result',
    score_signal_quality(F.col('normalized_signal'), F.col('missing_fraction'), F.col('sampling_rate'))
)

quality_df = quality_result_df.select(
    'patient_id', 'window_id', 'timestamp', 'cardiovascular_status', 'systolic_bp', 'diastolic_bp',
    'missing_fraction', 'signal_std', F.col('quality_result.*')
)

quality_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cardio_signal_quality')

print('Saved Delta table: cardio_signal_quality')
display(quality_df.orderBy(F.desc('signal_quality_score')).limit(10))
display(quality_df.select('signal_quality_score', 'missing_score', 'variance_score', 'peak_consistency_score', 'noise_score').summary())
